In [0]:
-- ============================================
-- BRONZE 层: 增量数据加载 (使用 MERGE)
-- ============================================

-- 1. 创建临时视图连接 MySQL (使用 Databricks 的 MySQL 连接器)
CREATE OR REPLACE TEMPORARY VIEW mysql_sku_info_latest AS
SELECT 
    id          ,
    spu_id        ,
    price,
    sku_name,
    sku_desc,
    weight,
    tm_id,
    category3_id,
    sku_default_img, 
    is_sale,
    create_time ,
    operate_time,
    CURRENT_TIMESTAMP() as _bronze_load_ts,
    'mysql_production' as _source_system,
    'sku_info' as _source_table
FROM awsmysql_catalog.gmall.sku_info
WHERE create_time > (
    SELECT COALESCE(MAX(create_time), '1900-01-01')
    FROM aws3.bronze.sku_info
);
---所以要定期-- 清理超过保留期限的文件（默认清理7天前的）我的表设置一天
--VACUUM your_table_name;实际数据在云上，但对DB，已经没了


-- 2. MERGE 到 Bronze 层
MERGE INTO aws3.bronze.sku_info AS target
USING mysql_sku_info_latest AS source
ON target.id = source.id 
   AND target._source_system = source._source_system
   AND target.create_time = source.create_time
WHEN NOT MATCHED THEN
INSERT (
    _bronze_load_ts,
    _bronze_load_id,
    _source_system,
    _source_table,
    id          ,
    spu_id        ,
    price,
    sku_name,
    sku_desc,
    weight,
    tm_id,
    category3_id,
    sku_default_img, 
    is_sale,
    create_time ,
    operate_time
)
VALUES (
    source._bronze_load_ts,
    UUID() ,--as _bronze_load_id,
    source._source_system,
    source._source_table,
    source.id          ,
    source.spu_id        ,
    source.price,
    source.sku_name,
    source.sku_desc,
    source.weight,
    source.tm_id,
    source.category3_id,
    source.sku_default_img,
    source.is_sale,
    source.create_time ,
    source.operate_time
);